In [98]:
STORAGE_ACCOUNT = "amazondatalake60307052"
ACCOUNT_KEY     = "rbXHUr9voqcrJI5eLoNEFzc4ULCNRK1qI7Ogtmh2gQrGGkLnR9uDGTEN2txtGTxC43EZpQKwse9K+AStPfT+KA=="

NEW_BASE    = "wasbs://lab5@" + STORAGE_ACCOUNT + ".blob.core.windows.net"
RAW_PATH    = NEW_BASE + "/raw"
SILVER_PATH = NEW_BASE + "/silver"
GOLD_PATH   = NEW_BASE + "/gold"

spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT}.blob.core.windows.net",
    ACCOUNT_KEY
)

print(f"RAW    → {RAW_PATH}")
print(f"SILVER → {SILVER_PATH}")
print(f"GOLD   → {GOLD_PATH}")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 153, Finished, Available, Finished)

RAW    → wasbs://lab5@amazondatalake60307052.blob.core.windows.net/raw
SILVER → wasbs://lab5@amazondatalake60307052.blob.core.windows.net/silver
GOLD   → wasbs://lab5@amazondatalake60307052.blob.core.windows.net/gold


In [99]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import io

STORAGE_ACCOUNT = "amazondatalake60307052"
ACCOUNT_KEY     = "rbXHUr9voqcrJI5eLoNEFzc4ULCNRK1qI7Ogtmh2gQrGGkLnR9uDGTEN2txtGTxC43EZpQKwse9K+AStPfT+KA=="
CONTAINER       = "lab5"            

blob_service = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
    credential=ACCOUNT_KEY
)

def read_raw(filename):
    data = blob_service.get_blob_client(
        container=CONTAINER, blob=f"raw/{filename}"
    ).download_blob().readall()
    return io.BytesIO(data)

def read_blob(path):
    data = blob_service.get_blob_client(
        container=CONTAINER, blob=path
    ).download_blob().readall()
    return io.BytesIO(data)

def save_parquet_to_blob(path, df):
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    blob_service.get_blob_client(
        container=CONTAINER, blob=path
    ).upload_blob(buf, overwrite=True)
    print(f"💾 Saved → {path}  ({len(df)} rows)")

container_client = blob_service.get_container_client(CONTAINER)
print("📦 Files in lab5:")
for blob in container_client.list_blobs():
    print(f"   📄 {blob.name}  ({blob.size} bytes)")


StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 154, Finished, Available, Finished)

📦 Files in lab5:
   📄 gold  (0 bytes)
   📄 gold/features_filtered_FD001.parquet  (3801813 bytes)
   📄 gold/features_final_FD001.parquet  (1638640 bytes)
   📄 gold/features_train_FD001.parquet  (14182585 bytes)
   📄 gold/final_report.json  (1427 bytes)
   📄 gold/scaler_FD001.pkl  (1583 bytes)
   📄 gold/test_FD001.parquet  (419149 bytes)
   📄 gold/train_FD001.parquet  (631510 bytes)
   📄 lab5  (0 bytes)
   📄 lab5/gold  (0 bytes)
   📄 lab5/gold/features_filtered_FD001.parquet  (3923968 bytes)
   📄 lab5/gold/features_train_FD001.parquet  (14182585 bytes)
   📄 raw  (0 bytes)
   📄 raw/RUL_FD001.txt  (429 bytes)
   📄 raw/test_FD001.txt  (2228855 bytes)
   📄 raw/train_FD001.txt  (3515356 bytes)
   📄 silver  (0 bytes)
   📄 silver/test_FD001.parquet  (354123 bytes)
   📄 silver/train_FD001.parquet  (537004 bytes)


In [100]:
COLUMN_NAMES = (
    ["engine_id", "cycle"] +
    [f"op_setting_{i}" for i in range(1, 4)] +
    [f"sensor_{i}"     for i in range(1, 22)]
)

print("📥 Reading files...")

train_df = pd.read_csv(read_raw("train_FD001.txt"),
                       sep=r"\s+", header=None, names=COLUMN_NAMES)

test_df  = pd.read_csv(read_raw("test_FD001.txt"),
                       sep=r"\s+", header=None, names=COLUMN_NAMES)

rul_df   = pd.read_csv(read_raw("RUL_FD001.txt"),
                       sep=r"\s+", header=None, names=["RUL"])

print(f"✅ Train : {train_df.shape}")
print(f"✅ Test  : {test_df.shape}")
print(f"✅ RUL   : {rul_df.shape}")
train_df.head()

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 155, Finished, Available, Finished)

📥 Reading files...
✅ Train : (20631, 26)
✅ Test  : (13096, 26)
✅ RUL   : (100, 1)


,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [101]:
# ════════════════════════════════════════════════════════
# STAGE 1 — RAW → SILVER  
# ════════════════════════════════════════════════════════

max_cycle = (train_df.groupby("engine_id")["cycle"]
             .max().reset_index()
             .rename(columns={"cycle": "max_cycle"}))

train_df = train_df.merge(max_cycle, on="engine_id")
train_df["RUL"] = train_df["max_cycle"] - train_df["cycle"]
train_df.drop(columns=["max_cycle"], inplace=True)

rul_df["engine_id"] = range(1, len(rul_df) + 1)
last_cycle = (test_df.groupby("engine_id")["cycle"]
              .max().reset_index()
              .rename(columns={"cycle": "max_cycle"}))
last_cycle = last_cycle.merge(rul_df, on="engine_id")
test_df = test_df.merge(last_cycle, on="engine_id")
test_df["RUL"] = test_df["RUL"] + (test_df["max_cycle"] - test_df["cycle"])
test_df.drop(columns=["max_cycle"], inplace=True)

print("✅ RUL added successfully")
print(f"   Train shape : {train_df.shape}")
print(f"   Test shape  : {test_df.shape}")
print(f"   RUL range   : {train_df['RUL'].min()} – {train_df['RUL'].max()} cycles")
train_df[["engine_id", "cycle", "RUL"]].head(8)

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 156, Finished, Available, Finished)

✅ RUL added successfully
   Train shape : (20631, 27)
   Test shape  : (13096, 27)
   RUL range   : 0 – 361 cycles


,engine_id,cycle,RUL
0,1,1,191
1,1,2,190
2,1,3,189
3,1,4,188
4,1,5,187
5,1,6,186
6,1,7,185
7,1,8,184


In [102]:
def save_parquet_to_blob(path, df):
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    blob_service.get_blob_client(
        container=CONTAINER, blob=path
    ).upload_blob(buf, overwrite=True)
    print(f"💾 Saved → {path}  ({len(df)} rows)")

print("\n📤 Writing Silver layer...")
save_parquet_to_blob("silver/train_FD001.parquet", train_df)
save_parquet_to_blob("silver/test_FD001.parquet",  test_df)
print("\n✅ Silver layer complete!")


StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 157, Finished, Available, Finished)


📤 Writing Silver layer...
💾 Saved → silver/train_FD001.parquet  (20631 rows)
💾 Saved → silver/test_FD001.parquet  (13096 rows)

✅ Silver layer complete!


In [103]:
# ════════════════════════════════════════════════════════
# STAGE 2 — SILVER → GOLD  (Scaling + Drop constant sensors)
# ════════════════════════════════════════════════════════
from sklearn.preprocessing import StandardScaler
import joblib

DROP_SENSORS = ["sensor_1","sensor_5","sensor_6",
                "sensor_10","sensor_16","sensor_18","sensor_19"]

FEATURE_COLS = [c for c in train_df.columns
                if (c.startswith("sensor_") or c.startswith("op_setting_"))
                and c not in DROP_SENSORS]

print(f"📊 Features before drop : {len([c for c in train_df.columns if c.startswith('sensor_')])}")
print(f"📊 Features after drop  : {len(FEATURE_COLS)}")
print(f"📊 Dropped sensors      : {DROP_SENSORS}")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 158, Finished, Available, Finished)

📊 Features before drop : 21
📊 Features after drop  : 17
📊 Dropped sensors      : ['sensor_1', 'sensor_5', 'sensor_6', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']


In [104]:
scaler = StandardScaler()

train_gold = train_df.copy()
test_gold  = test_df.copy()

train_gold[FEATURE_COLS] = scaler.fit_transform(train_df[FEATURE_COLS])
test_gold[FEATURE_COLS]  = scaler.transform(test_df[FEATURE_COLS])

print("✅ Scaling complete")
print(f"   Mean  (after scale): {train_gold[FEATURE_COLS].mean().mean():.6f}")
print(f"   Std   (after scale): {train_gold[FEATURE_COLS].std().mean():.6f} ")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 159, Finished, Available, Finished)

✅ Scaling complete
   Mean  (after scale): 0.000000
   Std   (after scale): 0.941199 


In [105]:
print("\n📤 Writing Gold layer...")

# ← احذف blob_service, CONTAINER, و lab5/
save_parquet_to_blob("gold/train_FD001.parquet", train_gold)
save_parquet_to_blob("gold/test_FD001.parquet",  test_gold)

# ← احذف lab5/ من المسار
scaler_buf = io.BytesIO()
joblib.dump(scaler, scaler_buf)
scaler_buf.seek(0)
blob_service.get_blob_client(
    container=CONTAINER, blob="gold/scaler_FD001.pkl"
).upload_blob(scaler_buf, overwrite=True)

print("💾 Saved → gold/scaler_FD001.pkl")
print("\n✅ Gold layer complete!")


StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 160, Finished, Available, Finished)


📤 Writing Gold layer...
💾 Saved → gold/train_FD001.parquet  (20631 rows)
💾 Saved → gold/test_FD001.parquet  (13096 rows)
💾 Saved → gold/scaler_FD001.pkl

✅ Gold layer complete!


In [106]:
# ════════════════════════════════════════════════════════
# STAGE 3 — GOLD → tsfresh Feature Extraction
# ════════════════════════════════════════════════════════
%pip install tsfresh --quiet

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 166, Finished, Available, Finished)


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [107]:
from azure.storage.blob import BlobServiceClient
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import io, joblib, time

STORAGE_ACCOUNT = "amazondatalake60307052"
ACCOUNT_KEY     = "rbXHUr9voqcrJI5eLoNEFzc4ULCNRK1qI7Ogtmh2gQrGGkLnR9uDGTEN2txtGTxC43EZpQKwse9K+AStPfT+KA=="
CONTAINER       = "lab5"           

blob_service = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
    credential=ACCOUNT_KEY
)

def read_blob(path):
    data = blob_service.get_blob_client(
        container=CONTAINER, blob=path).download_blob().readall()
    return io.BytesIO(data)

def save_parquet_to_blob(path, df):
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    blob_service.get_blob_client(
        container=CONTAINER, blob=path).upload_blob(buf, overwrite=True)
    print(f"💾 Saved → {path}  ({len(df)} rows)")

print("📥 Loading Gold layer...")
train_gold = pd.read_parquet(read_blob("gold/train_FD001.parquet"))
test_gold  = pd.read_parquet(read_blob("gold/test_FD001.parquet"))

DROP_SENSORS = ["sensor_1","sensor_5","sensor_6",
                "sensor_10","sensor_16","sensor_18","sensor_19"]

FEATURE_COLS   = [c for c in train_gold.columns
                  if (c.startswith("sensor_") or c.startswith("op_setting_"))
                  and c not in DROP_SENSORS]
ACTIVE_SENSORS = [c for c in FEATURE_COLS if c.startswith("sensor_")]

print(f"✅ Train gold : {train_gold.shape}")
print(f"✅ Test gold  : {test_gold.shape}")
print(f"✅ Features   : {len(FEATURE_COLS)}")
print("\n🚀 Ready to continue!")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 172, Finished, Available, Finished)

📥 Loading Gold layer...
✅ Train gold : (20631, 27)
✅ Test gold  : (13096, 27)
✅ Features   : 17

🚀 Ready to continue!


In [108]:
# ════════════════════════════════════════════════════════
# STAGE 3 — tsfresh Feature Extraction
# ════════════════════════════════════════════════════════
from tsfresh import extract_features
from tsfresh.feature_extraction import MinimalFCParameters  # ← تغيّر
from tsfresh.utilities.dataframe_functions import impute

WINDOW_SIZE    = 30
ACTIVE_SENSORS = [c for c in FEATURE_COLS if c.startswith("sensor_")]

print(f"⚙️  Config:")
print(f"   Window size    : {WINDOW_SIZE} cycles")
print(f"   Active sensors : {len(ACTIVE_SENSORS)}")
print(f"   Train rows     : {len(train_gold)}")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 173, Finished, Available, Finished)

⚙️  Config:
   Window size    : 30 cycles
   Active sensors : 14
   Train rows     : 20631


In [109]:
from tsfresh.feature_extraction import MinimalFCParameters
import time

print("🔄 Building rolling windows...")
t0 = time.time()

rows = []
for eng_id, grp in train_gold.groupby("engine_id"):
    grp = grp.sort_values("cycle").reset_index(drop=True)
    for end_idx in range(len(grp)):
        start_idx = max(0, end_idx - WINDOW_SIZE + 1)
        window = grp.iloc[start_idx:end_idx+1].copy()
        window["id"]   = eng_id * 10_000 + end_idx
        window["time"] = range(len(window))
        rows.append(window)

ts_df = pd.concat(rows, ignore_index=True)
ts_df = ts_df[["id", "time"] + ACTIVE_SENSORS]
print(f"✅ Done in {time.time()-t0:.1f}s | Shape: {ts_df.shape}")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 174, Finished, Available, Finished)

🔄 Building rolling windows...
✅ Done in 15.7s | Shape: (575430, 16)


In [110]:
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import impute

t0 = time.time()
X_features = extract_features(
    ts_df,
    column_id="id",
    column_sort="time",
    default_fc_parameters=MinimalFCParameters(),
    n_jobs=1,
    impute_function=impute,
    disable_progressbar=False,
)
elapsed = time.time() - t0
print(f"✅ Features: {X_features.shape[1]} in {elapsed/60:.1f} min")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 175, Finished, Available, Finished)

Feature Extraction: 100%|██████████| 288834/288834 [01:18<00:00, 3687.87it/s]


✅ Features: 140 in 1.4 min


In [111]:
import json
from datetime import datetime

y_train = (train_gold
           .assign(id=lambda d: d["engine_id"] * 10_000 +
                   d.groupby("engine_id").cumcount())
           .set_index("id")["RUL"])

X_features["RUL"] = y_train.reindex(X_features.index)

save_parquet_to_blob(
    "gold/features_train_FD001.parquet",
    X_features.reset_index()
)

timing_report = {
    "extraction_method"  : "MinimalFCParameters",
    "window_size"        : 30,
    "n_sensors"          : 14,
    "features_extracted" : X_features.shape[1],
    "rows"               : X_features.shape[0],
    "extraction_time_min": round(elapsed / 60, 2),
    "timestamp"          : datetime.now().isoformat(),
}
print(json.dumps(timing_report, indent=2))
print("✅ Features saved to Gold!")


StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 176, Finished, Available, Finished)

💾 Saved → gold/features_train_FD001.parquet  (20631 rows)
{
  "extraction_method": "MinimalFCParameters",
  "window_size": 30,
  "n_sensors": 14,
  "features_extracted": 141,
  "rows": 20631,
  "extraction_time_min": 1.38,
  "timestamp": "2026-03-12T07:57:45.895465"
}
✅ Features saved to Gold!


In [112]:
# ════════════════════════════════════════════════════════
# STAGE 4 — Filter-based Feature Selection
# ════════════════════════════════════════════════════════
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
import time

# ── preparing X and y ───────────────────────────────────────────
y_train = X_features["RUL"].dropna()
X       = X_features.drop(columns=["RUL"]).loc[y_train.index]

print(f"📊 Starting features : {X.shape[1]}")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 177, Finished, Available, Finished)

📊 Starting features : 140


In [113]:
# ── Stage 1: Variance Threshold ───────────────────────────
t0  = time.time()
sel = VarianceThreshold(threshold=0.01)
X_var     = sel.fit_transform(X)
kept_cols = X.columns[sel.get_support()]
X_var     = pd.DataFrame(X_var, columns=kept_cols, index=X.index)

print(f"✅ After Variance filter  : {X_var.shape[1]} features  "
      f"(dropped {X.shape[1] - X_var.shape[1]})")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 178, Finished, Available, Finished)

✅ After Variance filter  : 121 features  (dropped 19)


In [114]:
# ── Stage 2: Correlation Filter ───────────────────────────
corr_matrix = X_var.corr().abs()
upper       = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
drop_cols   = [c for c in upper.columns if any(upper[c] > 0.95)]
X_corr      = X_var.drop(columns=drop_cols)

print(f"✅ After Correlation filter: {X_corr.shape[1]} features  "
      f"(dropped {X_var.shape[1] - X_corr.shape[1]})")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 179, Finished, Available, Finished)

✅ After Correlation filter: 47 features  (dropped 74)


In [115]:
# ── Stage 3: Mutual Information Top-50 ────────────────────
mi      = mutual_info_regression(X_corr, y_train, random_state=42)
mi_series = pd.Series(mi, index=X_corr.columns).sort_values(ascending=False)
top_cols  = mi_series.head(50).index
X_filtered = X_corr[top_cols]

elapsed = time.time() - t0
print(f"✅ After MI filter        : {X_filtered.shape[1]} features  "
      f"(top 50 by mutual info)")
print(f"\n📊 Filter Summary:")
print(f"   Input    : {X.shape[1]} features")
print(f"   Output   : {X_filtered.shape[1]} features")
print(f"   Removed  : {X.shape[1] - X_filtered.shape[1]} features")
print(f"   Time     : {elapsed:.1f}s")

print(f"\n🏆 Top 10 most informative features:")
for i, (col, score) in enumerate(mi_series.head(10).items(), 1):
    print(f"   {i:2}. {col:<50} MI={score:.4f}")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 180, Finished, Available, Finished)

✅ After MI filter        : 47 features  (top 50 by mutual info)

📊 Filter Summary:
   Input    : 140 features
   Output   : 47 features
   Removed  : 93 features
   Time     : 16.9s

🏆 Top 10 most informative features:
    1. sensor_2__maximum                                  MI=0.6671
    2. sensor_7__maximum                                  MI=0.6542
    3. sensor_15__maximum                                 MI=0.6406
    4. sensor_3__maximum                                  MI=0.6379
    5. sensor_21__minimum                                 MI=0.6281
    6. sensor_2__sum_values                               MI=0.6257
    7. sensor_11__absolute_maximum                        MI=0.6046
    8. sensor_20__minimum                                 MI=0.5912
    9. sensor_20__absolute_maximum                        MI=0.5381
   10. sensor_15__minimum                                 MI=0.5305


In [116]:
# ── Saving Filtered Features  ──────────────
X_filtered = X_filtered.copy()   
X_filtered["RUL"] = y_train

save_parquet_to_blob(
    "gold/features_filtered_FD001.parquet",
    X_filtered.reset_index()
)
print("✅ Filtered features saved to Gold!")



StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 181, Finished, Available, Finished)

💾 Saved → gold/features_filtered_FD001.parquet  (20631 rows)
✅ Filtered features saved to Gold!


In [117]:
# ════════════════════════════════════════════════════════
# STAGE 5 
# ════════════════════════════════════════════════════════
%pip install tsfresh deap lightgbm xgboost --quiet

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 187, Finished, Available, Finished)


[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [118]:
from azure.storage.blob import BlobServiceClient
from deap import base, creator, tools, algorithms
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
from sklearn.feature_selection import VarianceThreshold, mutual_info_regression
from sklearn.preprocessing import StandardScaler
from tsfresh import extract_features
from tsfresh.feature_extraction import MinimalFCParameters
from tsfresh.utilities.dataframe_functions import impute
import pandas as pd
import numpy as np
import io, time, random, joblib

STORAGE_ACCOUNT = "amazondatalake60307052"
ACCOUNT_KEY     = "rbXHUr9voqcrJI5eLoNEFzc4ULCNRK1qI7Ogtmh2gQrGGkLnR9uDGTEN2txtGTxC43EZpQKwse9K+AStPfT+KA=="
CONTAINER       = "lab5"

blob_service = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
    credential=ACCOUNT_KEY
)

def read_blob(path):
    data = blob_service.get_blob_client(
        container=CONTAINER, blob=path).download_blob().readall()
    return io.BytesIO(data)

def save_parquet_to_blob(path, df):
    buf = io.BytesIO()
    df.to_parquet(buf, index=False)
    buf.seek(0)
    blob_service.get_blob_client(
        container=CONTAINER, blob=path).upload_blob(buf, overwrite=True)
    print(f"💾 Saved → {path}  ({len(df)} rows)")

print("✅ Setup complete")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 193, Finished, Available, Finished)

✅ Setup complete


In [119]:
print("📥 Loading Gold layer...")

train_gold = pd.read_parquet(read_blob("gold/train_FD001.parquet"))
test_gold  = pd.read_parquet(read_blob("gold/test_FD001.parquet"))

DROP_SENSORS   = ["sensor_1","sensor_5","sensor_6",
                  "sensor_10","sensor_16","sensor_18","sensor_19"]
FEATURE_COLS   = [c for c in train_gold.columns
                  if (c.startswith("sensor_") or c.startswith("op_setting_"))
                  and c not in DROP_SENSORS] 
ACTIVE_SENSORS = [c for c in FEATURE_COLS if c.startswith("sensor_")]

print(f"✅ Train : {train_gold.shape}")
print(f"✅ Test  : {test_gold.shape}")
print(f"✅ Features: {len(FEATURE_COLS)}")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 194, Finished, Available, Finished)

📥 Loading Gold layer...
✅ Train : (20631, 27)
✅ Test  : (13096, 27)
✅ Features: 17


In [120]:
WINDOW_SIZE = 30
print("🔄 Building rolling windows...")
t0 = time.time()

rows = []
for eng_id, grp in train_gold.groupby("engine_id"):
    grp = grp.sort_values("cycle").reset_index(drop=True)
    for end_idx in range(len(grp)):
        start_idx = max(0, end_idx - WINDOW_SIZE + 1)
        window = grp.iloc[start_idx:end_idx+1].copy()
        window["id"]   = eng_id * 10_000 + end_idx
        window["time"] = range(len(window))
        rows.append(window)

ts_df = pd.concat(rows, ignore_index=True)
ts_df = ts_df[["id", "time"] + ACTIVE_SENSORS]

print(f"✅ Done in {time.time()-t0:.1f}s")
print(f"   Shape: {ts_df.shape}")


StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 195, Finished, Available, Finished)

🔄 Building rolling windows...
✅ Done in 15.7s
   Shape: (575430, 16)


In [121]:
print("⚙️  Extracting features...")
t0 = time.time()

X_features = extract_features(
    ts_df,
    column_id="id",
    column_sort="time",
    default_fc_parameters=MinimalFCParameters(),
    n_jobs=1,
    impute_function=impute,
    disable_progressbar=False,
)

elapsed = time.time() - t0
print(f"✅ Extracted: {X_features.shape[1]} features in {elapsed/60:.1f} min")


y_map = (train_gold
         .assign(id=lambda d: d["engine_id"] * 10_000 +
                 d.groupby("engine_id").cumcount())
         .set_index("id")["RUL"])
X_features["RUL"] = y_map.reindex(X_features.index)

save_parquet_to_blob("gold/features_train_FD001.parquet",
                     X_features.reset_index())
print("✅ Features saved!")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 196, Finished, Available, Finished)

Feature Extraction: 100%|██████████| 288834/288834 [01:17<00:00, 3713.32it/s]


✅ Extracted: 140 features in 1.4 min
💾 Saved → gold/features_train_FD001.parquet  (20631 rows)
✅ Features saved!


In [122]:
y_train    = X_features["RUL"].dropna()
X          = X_features.drop(columns=["RUL"]).loc[y_train.index]

# Stage 1: Variance
sel   = VarianceThreshold(threshold=0.01)
X_var = pd.DataFrame(sel.fit_transform(X),
                     columns=X.columns[sel.get_support()],
                     index=X.index)
print(f"✅ After Variance    : {X_var.shape[1]}")

# Stage 2: Correlation
corr  = X_var.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop  = [c for c in upper.columns if any(upper[c] > 0.95)]
X_corr = X_var.drop(columns=drop)
print(f"✅ After Correlation : {X_corr.shape[1]}")

# Stage 3: Mutual Information
mi        = mutual_info_regression(X_corr, y_train, random_state=42)
mi_series = pd.Series(mi, index=X_corr.columns).sort_values(ascending=False)
X_filtered = X_corr[mi_series.head(50).index].copy()
X_filtered["RUL"] = y_train.values
print(f"✅ After MI (top 50) : {X_filtered.shape[1]-1}")

save_parquet_to_blob("gold/features_filtered_FD001.parquet",
                     X_filtered.reset_index(drop=True))

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 197, Finished, Available, Finished)

✅ After Variance    : 121
✅ After Correlation : 47
✅ After MI (top 50) : 47
💾 Saved → gold/features_filtered_FD001.parquet  (20631 rows)


In [123]:
from sklearn.tree import DecisionTreeRegressor

y_train    = X_filtered["RUL"].dropna()
X_ga       = X_filtered.drop(columns=["RUL"]).loc[y_train.index]
X_arr      = X_ga.values
y_arr      = y_train.values
n_features = X_ga.shape[1]

random.seed(42); np.random.seed(42)

if "FitnessMin" not in creator.__dict__:
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
toolbox.register("attr_bool",  random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat,
                 creator.Individual, toolbox.attr_bool, n=n_features)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

fast_model = DecisionTreeRegressor(max_depth=5, random_state=42)

def evaluate(individual):
    selected = [i for i, bit in enumerate(individual) if bit == 1]
    if len(selected) == 0:
        return (1e9,)
    scores = cross_val_score(
        fast_model, X_arr[:, selected], y_arr,
        cv=2, scoring="neg_root_mean_squared_error")
    return (-scores.mean() + 0.01 * len(selected) / n_features,)

toolbox.register("evaluate", evaluate)
toolbox.register("mate",     tools.cxTwoPoint)
toolbox.register("mutate",   tools.mutFlipBit, indpb=0.05)
toolbox.register("select",   tools.selTournament, tournsize=3)

print("🧬 Running Fast GA...")
t0    = time.time()
pop   = toolbox.population(n=20)
hof   = tools.HallOfFame(1)
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("min", np.min)
stats.register("avg", np.mean)

pop, log = algorithms.eaSimple(
    pop, toolbox, cxpb=0.6, mutpb=0.2, ngen=10,
    stats=stats, halloffame=hof, verbose=True)

ga_time       = time.time() - t0
selected_cols = [X_ga.columns[i]
                 for i, bit in enumerate(hof[0]) if bit == 1]

print(f"\n✅ GA Complete in {ga_time/60:.1f} min")
print(f"   Selected: {len(selected_cols)} / {n_features} features")

X_final = X_ga[selected_cols].copy()
X_final["RUL"] = y_train.values
save_parquet_to_blob("gold/features_final_FD001.parquet",
                     X_final.reset_index(drop=True))

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 198, Finished, Available, Finished)

🧬 Running Fast GA...
gen	nevals	min    	avg    
0  	20    	46.7877	49.6216
1  	12    	46.7877	47.8033
2  	15    	46.5452	47.2789
3  	16    	46.4554	47.1739
4  	15    	46.4554	47.0924
5  	16    	46.4554	47.2576
6  	11    	46.4554	46.7253
7  	18    	46.0201	46.6149
8  	12    	46.0023	46.5341
9  	13    	46.0023	46.3055
10 	13    	46.0023	46.1942

✅ GA Complete in 0.6 min
   Selected: 17 / 47 features
💾 Saved → gold/features_final_FD001.parquet  (20631 rows)


In [124]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("📥 Building test features...")
rows = []
for eng_id, grp in test_gold.groupby("engine_id"):
    grp = grp.sort_values("cycle").reset_index(drop=True)
    for end_idx in range(len(grp)):
        start_idx = max(0, end_idx - WINDOW_SIZE + 1)
        window = grp.iloc[start_idx:end_idx+1].copy()
        window["id"]   = eng_id * 10_000 + end_idx
        window["time"] = range(len(window))
        rows.append(window)

ts_test = pd.concat(rows, ignore_index=True)
ts_test = ts_test[["id", "time"] + ACTIVE_SENSORS]

print("⚙️  Extracting test features...")
X_test_raw = extract_features(
    ts_test, column_id="id", column_sort="time",
    default_fc_parameters=MinimalFCParameters(),
    n_jobs=1, impute_function=impute, disable_progressbar=False)

y_test_map = (test_gold
              .assign(id=lambda d: d["engine_id"] * 10_000 +
                      d.groupby("engine_id").cumcount())
              .set_index("id")["RUL"])

y_test = y_test_map.reindex(X_test_raw.index).dropna()
X_test = X_test_raw.reindex(columns=selected_cols).fillna(0).loc[y_test.index]

# ──Training the model ─────────────────────────────────
y_train_final = X_final["RUL"]
X_train_final = X_final.drop(columns=["RUL"])

print("\n🤖 Training Random Forest...")
t0 = time.time()
rf_final = RandomForestRegressor(
    n_estimators=200, n_jobs=1, random_state=42)
rf_final.fit(X_train_final, y_train_final)
print(f"✅ Trained in {time.time()-t0:.1f}s")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 199, Finished, Available, Finished)

📥 Building test features...

🤖 Training Random Forest...
✅ Trained in 79.2s


Feature Extraction: 100%|██████████| 183344/183344 [00:49<00:00, 3671.16it/s]


In [127]:
# ════════════════════════════════════════════════════════
# CELL 9 — Evaluate
# ════════════════════════════════════════════════════════
preds = rf_final.predict(X_test)
rmse  = np.sqrt(mean_squared_error(y_test, preds))
mae   = mean_absolute_error(y_test, preds)
r2    = r2_score(y_test, preds)

print("=" * 45)
print("📊 FINAL RESULTS — Lab 5")
print("=" * 45)
print(f"   RMSE              : {rmse:.3f}")
print(f"   MAE               : {mae:.3f}")
print(f"   R²                : {r2:.3f}")
print(f"   Features (GA)     : {len(selected_cols)} / 47")
print(f"   Features (raw)    : 140")
print("=" * 45)
print("\n⏱️  Pipeline Runtime Summary:")
print(f"   tsfresh (train)   : 1.4 min")
print(f"   Filter selection  : < 1 min")
print(f"   GA selection      : 0.6 min")
print(f"   Model training    : 80.2 sec")
print(f"   tsfresh (test)    : 0.8 min")
print(f"   Total             : ~5 min ✅")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 202, Finished, Available, Finished)

📊 FINAL RESULTS — Lab 5
   RMSE              : 47.605
   MAE               : 36.242
   R²                : 0.348
   Features (GA)     : 17 / 47
   Features (raw)    : 140

⏱️  Pipeline Runtime Summary:
   tsfresh (train)   : 1.4 min
   Filter selection  : < 1 min
   GA selection      : 0.6 min
   Model training    : 80.2 sec
   tsfresh (test)    : 0.8 min
   Total             : ~5 min ✅


In [125]:
# ════════════════════════════════════════════════════════
# CELL 10 — RMSE
# ════════════════════════════════════════════════════════
try:
    import xgboost as xgb
    import lightgbm as lgb
    HAS_XGB = True
    HAS_LGB = True
except:
    HAS_XGB = False
    HAS_LGB = False

results = {}

# ── 1. Random Forest
results["RandomForest"] = {"rmse": 47.605, "mae": 36.242, "r2": 0.348}

# ── 2. XGBoost ────────────────────────────────────────────
if HAS_XGB:
    print("🤖 Training XGBoost...")
    t0 = time.time()
    xgb_model = xgb.XGBRegressor(
        n_estimators=300, learning_rate=0.05,
        max_depth=6, n_jobs=1,
        random_state=42, verbosity=0)
    xgb_model.fit(X_train_final, y_train_final,
                  eval_set=[(X_test, y_test)],
                  verbose=False)
    preds_xgb = xgb_model.predict(X_test)
    results["XGBoost"] = {
        "rmse": np.sqrt(mean_squared_error(y_test, preds_xgb)),
        "mae" : mean_absolute_error(y_test, preds_xgb),
        "r2"  : r2_score(y_test, preds_xgb),
        "time": time.time() - t0
    }
    print(f"✅ XGBoost RMSE: {results['XGBoost']['rmse']:.3f}")

# ── 3. LightGBM ───────────────────────────────────────────
if HAS_LGB:
    print("🤖 Training LightGBM...")
    t0 = time.time()
    lgb_model = lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05,
        max_depth=6, n_jobs=1,
        random_state=42, verbose=-1)
    lgb_model.fit(X_train_final, y_train_final)
    preds_lgb = lgb_model.predict(X_test)
    results["LightGBM"] = {
        "rmse": np.sqrt(mean_squared_error(y_test, preds_lgb)),
        "mae" : mean_absolute_error(y_test, preds_lgb),
        "r2"  : r2_score(y_test, preds_lgb),
        "time": time.time() - t0
    }
    print(f"✅ LightGBM RMSE: {results['LightGBM']['rmse']:.3f}")

# ── 4. RF  ───────────────────────────────────────────
print("🤖 Training Tuned Random Forest...")
t0 = time.time()
rf_tuned = RandomForestRegressor(
    n_estimators=500, max_depth=15,
    min_samples_leaf=2, n_jobs=1, random_state=42)
rf_tuned.fit(X_train_final, y_train_final)
preds_rf2 = rf_tuned.predict(X_test)
results["RF_Tuned"] = {
    "rmse": np.sqrt(mean_squared_error(y_test, preds_rf2)),
    "mae" : mean_absolute_error(y_test, preds_rf2),
    "r2"  : r2_score(y_test, preds_rf2),
    "time": time.time() - t0
}
print(f"✅ RF Tuned RMSE: {results['RF_Tuned']['rmse']:.3f}")

# ── results────────────────────────────────────────
print("\n" + "=" * 50)
print("📊 MODEL COMPARISON")
print("=" * 50)
print(f"{'Model':<20} {'RMSE':>8} {'MAE':>8} {'R²':>8}")
print("-" * 50)
for name, r in sorted(results.items(), key=lambda x: x[1]["rmse"]):
    print(f"{name:<20} {r['rmse']:>8.3f} {r['mae']:>8.3f} {r['r2']:>8.3f}")
print("=" * 50)

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 200, Finished, Available, Finished)

🤖 Training XGBoost...
✅ XGBoost RMSE: 46.278
🤖 Training LightGBM...
✅ LightGBM RMSE: 46.549
🤖 Training Tuned Random Forest...
✅ RF Tuned RMSE: 47.454

📊 MODEL COMPARISON
Model                    RMSE      MAE       R²
--------------------------------------------------
XGBoost                46.278   35.062    0.384
LightGBM               46.549   35.631    0.377
RF_Tuned               47.454   36.076    0.353
RandomForest           47.605   36.242    0.348


In [126]:
# ════════════════════════════════════════════════════════
# Rebort
# ════════════════════════════════════════════════════════
import json
from datetime import datetime

report = {
    "lab"      : "Lab 5 - Predictive Maintenance",
    "dataset"  : "NASA C-MAPSS FD001",
    "timestamp": datetime.now().isoformat(),

    "pipeline_steps": {
        "1_data_loading"       : "✅ train(20631) test(13096) RUL(100)",
        "2_preprocessing"      : "✅ Raw→Silver→Gold | dropped 7 sensors",
        "3_tsfresh_extraction" : "✅ 140 features | 1.4 min (MinimalFCParameters)",
        "4_filter_selection"   : "✅ 140→121→47 features (Variance+Corr+MI)",
        "5_genetic_algorithm"  : "✅ 47→17 features | 0.6 min | 10 gen pop=20",
        "6_model_training"     : "✅ XGBoost 300 estimators | 80s",
    },

    "results": {
        "XGBoost"     : {"rmse": 46.278, "mae": 35.062, "r2": 0.384},
        "LightGBM"    : {"rmse": 46.549, "mae": 35.631, "r2": 0.377},
        "RF_Tuned"    : {"rmse": 47.454, "mae": 36.076, "r2": 0.353},
        "RandomForest": {"rmse": 47.605, "mae": 36.242, "r2": 0.348},
    },

    "best_model": {
        "name"    : "XGBoost",
        "rmse"    : 46.278,
        "mae"     : 35.062,
        "r2"      : 0.384,
        "features": 17,
    },s

    "runtime_summary": {
        "tsfresh_train_min" : 1.4,
        "filter_selection_min": 0.5,
        "ga_selection_min"  : 0.6,
        "model_training_sec": 80.2,
        "tsfresh_test_min"  : 0.8,
        "total_min"         : 5.0,
    },

    "feature_reduction": {
        "raw_features"      : 140,
        "after_filter"      : 47,
        "after_ga"          : 17,
        "reduction_pct"     : "87.8%",
    }
}

report_buf = io.BytesIO(json.dumps(report, indent=2).encode())
blob_service.get_blob_client(
    container=CONTAINER,
    blob="gold/final_report.json"
).upload_blob(report_buf, overwrite=True)

print("=" * 55)
print("🏁 LAB 5 COMPLETE — FINAL SUMMARY")
print("=" * 55)
print(f"  Dataset          : NASA C-MAPSS FD001")
print(f"  Best Model       : XGBoost")
print(f"  RMSE             : 46.278  🏆")
print(f"  MAE              : 35.062")
print(f"  R²               : 0.384")
print(f"  Features used    : 17 / 140  (87.8% reduction)")
print(f"  Total runtime    : ~5 minutes  ⚡")
print("=" * 55)
print("\n📦 Files saved in lab5/gold/:")
print("   ├── train_FD001.parquet")
print("   ├── test_FD001.parquet")
print("   ├── features_train_FD001.parquet")
print("   ├── features_filtered_FD001.parquet")
print("   ├── features_final_FD001.parquet")
print("   ├── scaler_FD001.pkl")
print("   └── final_report.json")
print("\n✅ Lab 5 pipeline complete!")

StatementMeta(a96e25bb-24aa-4631-8494-418b48c8077c, 1, 201, Finished, Available, Finished)

🏁 LAB 5 COMPLETE — FINAL SUMMARY
  Dataset          : NASA C-MAPSS FD001
  Best Model       : XGBoost
  RMSE             : 46.278  🏆
  MAE              : 35.062
  R²               : 0.384
  Features used    : 17 / 140  (87.8% reduction)
  Total runtime    : ~5 minutes  ⚡

📦 Files saved in lab5/gold/:
   ├── train_FD001.parquet
   ├── test_FD001.parquet
   ├── features_train_FD001.parquet
   ├── features_filtered_FD001.parquet
   ├── features_final_FD001.parquet
   ├── scaler_FD001.pkl
   └── final_report.json

✅ Lab 5 pipeline complete!
